In [3]:
import os
os.getcwd()

'/Users/jason/Documents/GitHub/InterOptimus/Tutorial/LLM'

In [4]:
from InterOptimus.agents.llm_iomaker_job import BuildConfig, build_iomaker_flow_from_prompt

cfg = BuildConfig(
    api_key="sk-zk29a8909a9badfb2973536dfd5a1bf41c7696742c764c35",
    base_url="https://api.zhizengzeng.com/v1/",
    model="gpt-3.5-turbo",
    mp_api_key="LBPWGcTye2eanNBb7dtmTp2deXR4aF9E",  # 若本地无cif则需要
    output_flow_json="io_flow.json",
)

result = build_iomaker_flow_from_prompt(
    "双界面模型，没有真空层，不做vasp",
    cfg,
)

print(result["flow_json_path"])
print(result["structures_meta"])


✅ InterOptimus IOMaker job generated
📄 Flow JSON: /Users/jason/Documents/GitHub/InterOptimus/Tutorial/LLM/io_flow.json
📦 Structures source: local_cif
🧱 film.cif: /Users/jason/Documents/GitHub/InterOptimus/Tutorial/LLM/film.cif
🧱 substrate.cif: /Users/jason/Documents/GitHub/InterOptimus/Tutorial/LLM/substrate.cif

🔧 Parameter settings (LLM output normalized):
{
  "name": "IO_llm",
  "mode": "without_vacuum",
  "inputs": {
    "type": "local_cif",
    "film_cif": "film.cif",
    "substrate_cif": "substrate.cif"
  },
  "lattice_matching_settings": {
    "max_area": 25,
    "max_length_tol": 0.03,
    "max_angle_tol": 0.03,
    "film_max_miller": 3,
    "substrate_max_miller": 3,
    "film_millers": null,
    "substrate_millers": null
  },
  "structure_settings": {
    "termination_ftol": 0.15,
    "film_thickness": 8,
    "substrate_thickness": 8,
    "double_interface": true,
    "vacuum_over_film": 0
  },
  "optimization_settings": {
    "fmax": 0.05,
    "steps": 200,
    "device": "c

/Users/jason/Downloads/pymatgen-master/src/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 14 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


In [8]:
import subprocess
import shlex
from pathlib import Path

def submit_io_flow_via_ssh(
    host="xys@10.103.65.21",
    identity_file="~/.ssh/id_ed25519",
    local_io_flow_json="io_flow.json",
    remote_workdir="~/io_runs/si_sic_run1",
    remote_python="python",
    pre_cmd="source ~/.bashrc && conda activate atomate2",
):
    local_io_flow_json = Path(local_io_flow_json).resolve()
    if not local_io_flow_json.exists():
        raise FileNotFoundError(f"Local file not found: {local_io_flow_json}")

    identity_file = str(Path(identity_file).expanduser())
    ssh_opts = [
        "-i", identity_file,
        "-o", "IdentitiesOnly=yes",
        "-o", "PreferredAuthentications=publickey",
        "-o", "StrictHostKeyChecking=accept-new",
    ]

    # 远端建目录：不要把 ~ 包在引号里
    subprocess.run(["ssh", *ssh_opts, host, f"mkdir -p {remote_workdir}"], check=True)

    # 上传 io_flow.json
    subprocess.run(
        ["scp", *ssh_opts, str(local_io_flow_json), f"{host}:{remote_workdir.rstrip('/')}/io_flow.json"],
        check=True,
    )

    remote_script = r"""
from jobflow_remote.utils.examples import add
from jobflow_remote import submit_flow
from jobflow import Flow
from qtoolkit.core.data_objects import QResources
import json

with open('io_flow.json','r') as f:
    flow_json = json.load(f)

flow = Flow.from_dict(flow_json)

resources = QResources(
    nodes=1,
    processes_per_node=1,
    scheduler_kwargs={"partition": "standard", "cpus-per-task": 10},
)

print(submit_flow(flow, worker="std_worker", resources=resources, project="std"))
""".strip()

    cmd = f"""
set -e
cd {shlex.quote(remote_workdir)}
{pre_cmd}
{shlex.quote(remote_python)} - <<'PY'
{remote_script}
PY
""".strip()

    res = subprocess.run(
        ["ssh", *ssh_opts, host, "bash", "-lc", cmd],
        check=True,
        text=True,
        capture_output=True,
    )
    print(res.stdout)
    if res.stderr.strip():
        print("=== STDERR ===")
        print(res.stderr)

submit_io_flow_via_ssh()

xys@10.103.65.21: Permission denied (publickey,gssapi-keyex,gssapi-with-mic,password).


CalledProcessError: Command '['ssh', '-i', '/Users/jason/.ssh/id_ed25519', '-o', 'IdentitiesOnly=yes', '-o', 'PreferredAuthentications=publickey', '-o', 'StrictHostKeyChecking=accept-new', 'xys@10.103.65.21', 'mkdir -p ~/io_runs/si_sic_run1']' returned non-zero exit status 255.